In [114]:
import pandas as pd
import numpy as np
import json



In [115]:
midfielders = pd.read_csv(
    "data/processed/ midfielders_scored.csv"
)

forwards = pd.read_csv(
    "data/processed/ forwards_scored.csv"
)

defenders = pd.read_csv(
    "data/processed/ defenders_scored.csv"
)

position_data = {
    "Midfielder": midfielders,
    "Forward": forwards,
    "Defender": defenders
}

C:\Users\Titus\AppData\Local\Temp\ipykernel_27764\1818851210.py:1: DtypeWarning: Columns (0: weak_foot) have mixed types. Specify dtype option on import or set low_memory=False.
  midfielders = pd.read_csv(


In [116]:
allowed_factors = {
    "leadership",
    "decision_making",
    "work_rate",
    "tactical_fit",
    "press_resistance",
    "adaptability",
    "consistency",
    "injury_concern",
    "communication",
    "off_ball_movement"
}

In [117]:
factor_weights = {
    "leadership": 2.0,
    "decision_making": 2.0,
    "work_rate": 2.0,
    "tactical_fit": 3.0,
    "press_resistance": 2.5,
    "adaptability": 1.5,
    "consistency": 2.5,
    "injury_concern": 3.0,
    "communication": 1.5,
    "off_ball_movement": 2.0
}

In [118]:
example_llm_output = {
    "factors": [
        {
            "factor": "leadership",
            "direction": 1,
            "strength": 0.8,
            "confidence": 0.9
        },
        {
            "factor": "injury_concern",
            "direction": -1,
            "strength": 0.7,
            "confidence": 0.8
        }
    ]
}

In [119]:
def validate_factors(parsed_output):

    valid_factors = []

    for item in parsed_output.get(
        "factors",
        []
    ):

        factor = item.get("factor")

        if factor not in allowed_factors:
            continue

        direction = item.get(
            "direction",
            0
        )

        strength = item.get(
            "strength",
            0
        )

        confidence = item.get(
            "confidence",
            0
        )

        # Validate direction
        if direction not in [-1, 1]:
            continue

        # Keep strength/confidence between 0 and 1
        strength = min(
            max(float(strength), 0),
            1
        )

        confidence = min(
            max(float(confidence), 0),
            1
        )

        valid_factors.append({
            "factor": factor,
            "direction": direction,
            "strength": strength,
            "confidence": confidence
        })

    return valid_factors

In [120]:
mock_chatbot_output = {
    "factors": [

        {
            "factor": "leadership",
            "direction": 1,
            "strength": 0.9,
            "confidence": 0.95
        },

        {
            "factor": "work_rate",
            "direction": 1,
            "strength": 0.9,
            "confidence": 0.90
        },

        {
            "factor": "injury_concern",
            "direction": -1,
            "strength": 0.8,
            "confidence": 0.85
        }

    ]
}


validated = validate_factors(
    mock_chatbot_output
)

print(validated)


[{'factor': 'leadership', 'direction': 1, 'strength': 0.9, 'confidence': 0.95}, {'factor': 'work_rate', 'direction': 1, 'strength': 0.9, 'confidence': 0.9}, {'factor': 'injury_concern', 'direction': -1, 'strength': 0.8, 'confidence': 0.85}]


In [121]:
def calculate_context_adjustments(
    factors
):

    adjustments = []

    for item in factors:

        factor = item["factor"]

        weight = factor_weights[
            factor
        ]

        adjustment = (
            weight
            * item["direction"]
            * item["strength"]
            * item["confidence"]
        )

        adjustments.append({
            "factor": factor,
            "direction": item["direction"],
            "strength": item["strength"],
            "confidence": item["confidence"],
            "weight": weight,
            "adjustment": adjustment
        })

    return adjustments

In [122]:
adjustments = (
    calculate_context_adjustments(
        validated
    )
)

print(adjustments)

[{'factor': 'leadership', 'direction': 1, 'strength': 0.9, 'confidence': 0.95, 'weight': 2.0, 'adjustment': 1.71}, {'factor': 'work_rate', 'direction': 1, 'strength': 0.9, 'confidence': 0.9, 'weight': 2.0, 'adjustment': 1.62}, {'factor': 'injury_concern', 'direction': -1, 'strength': 0.8, 'confidence': 0.85, 'weight': 3.0, 'adjustment': -2.04}]


In [123]:
MAX_TOTAL_ADJUSTMENT = 10

def calculate_personalized_score(
    base_score,
    adjustments
):

    total_adjustment = sum(
        item["adjustment"]
        for item in adjustments
    )

    total_adjustment = np.clip(
        total_adjustment,
        -MAX_TOTAL_ADJUSTMENT,
        MAX_TOTAL_ADJUSTMENT
    )

    personalized_score = np.clip(
        base_score + total_adjustment,
        0,
        100
    )

    return (
        round(personalized_score, 1),
        round(total_adjustment, 2)
    )

In [124]:
#Player lookup function
def get_player_score(
    player_name,
    position,
    position_data,
    season=None
):

    dataframe = position_data[position]

    player_rows = dataframe[
        dataframe["player"].str.lower()
        == player_name.lower()
    ]

    if player_rows.empty:
        raise ValueError(
            f"{player_name} not found in {position} data."
        )

    if season is not None:

        season_rows = player_rows[
            player_rows["season"] == season
        ]

        if season_rows.empty:
            available_seasons = (
                player_rows["season"]
                .dropna()
                .sort_values()
                .unique()
            )

            raise ValueError(
                f"{player_name} does not have data "
                f"for season {season}. "
                f"Available seasons: {available_seasons}"
            )

        player_rows = season_rows

    player_row = (
        player_rows
        .sort_values(
            "season",
            ascending=False
        )
        .iloc[0]
    )

    return player_row[
        [
            "player",
            "team",
            "season",
            "player_positions",
            "archetype",
            "performance_score",
            "performance_percentile",
            "archetype_rank",
            "data_completeness",
            "score_confidence",
            "attacking_score",
            "creation_score",
            "progression_score",
            "possession_score",
            "defending_score"
        ]
    ]



In [125]:
def personalize_player_score(
    player_name,
    position,
    position_data,
    parsed_chatbot_output,
    season=None
):

    player = get_player_score(
        player_name,
        position,
        position_data,
        season=season
    )

    base_score = player[
        "performance_score"
    ]

    factors = validate_factors(
        parsed_chatbot_output
    )

    adjustments = (
        calculate_context_adjustments(
            factors
        )
    )

    personalized_score, total_adjustment = (
        calculate_personalized_score(
            base_score,
            adjustments
        )
    )

    return {
        "player": player_name,
        "position": position,
        "team": player["team"],
        "season": player["season"],
        "archetype": player["archetype"],
        "base_score": base_score,
        "personalized_score": personalized_score,
        "total_adjustment": total_adjustment,
        "adjustments": adjustments,
        "factors": adjustments
    }

In [126]:
def print_personalized_report(result):

    print(f"\nPlayer: {result['player']}")
    print(f"Team: {result['team']}")
    print(f"Season: {result['season']}")
    print(f"Archetype: {result['archetype']}")

    print("\n---------------------------")

    print(
        f"Baseline Score: "
        f"{result['base_score']}"
    )

    print(
        f"Personalized Score: "
        f"{result['personalized_score']}"
    )

    print(
        f"Total Adjustment: "
        f"{result['total_adjustment']:+.2f}"
    )

    print("\nContextual Factors:")

    for factor in result["factors"]:

        print(
            f"\n{factor['factor']}"
        )

        print(
            f"  Adjustment: "
            f"{factor['adjustment']:+.2f}"
        )

        print(
            f"  Strength: "
            f"{factor['strength']:.2f}"
        )

        print(
            f"  Confidence: "
            f"{factor['confidence']:.2f}"
        )

result = personalize_player_score(
    "Kevin De Bruyne",
    "Midfielder",
    position_data,
    mock_chatbot_output,
    season=2425
)

print_personalized_report(
    result
)

print(result)


Player: Kevin De Bruyne
Team: Manchester City
Season: 2425
Archetype: Creative / Progressive Midfielder

---------------------------
Baseline Score: 73.37662337662337
Personalized Score: 74.7
Total Adjustment: +1.29

Contextual Factors:

leadership
  Adjustment: +1.71
  Strength: 0.90
  Confidence: 0.95

work_rate
  Adjustment: +1.62
  Strength: 0.90
  Confidence: 0.90

injury_concern
  Adjustment: -2.04
  Strength: 0.80
  Confidence: 0.85
{'player': 'Kevin De Bruyne', 'position': 'Midfielder', 'team': 'Manchester City', 'season': np.int64(2425), 'archetype': 'Creative / Progressive Midfielder', 'base_score': np.float64(73.37662337662337), 'personalized_score': np.float64(74.7), 'total_adjustment': np.float64(1.29), 'adjustments': [{'factor': 'leadership', 'direction': 1, 'strength': 0.9, 'confidence': 0.95, 'weight': 2.0, 'adjustment': 1.71}, {'factor': 'work_rate', 'direction': 1, 'strength': 0.9, 'confidence': 0.9, 'weight': 2.0, 'adjustment': 1.62}, {'factor': 'injury_concern', 'd

# Two different users (Scouts)

In [127]:
scout_a = {
    "factors": [

        {
            "factor": "leadership",
            "direction": 1,
            "strength": 0.9,
            "confidence": 0.9
        },

        {
            "factor": "injury_concern",
            "direction": -1,
            "strength": 0.8,
            "confidence": 0.9
        }

    ]
}

scout_b = {
    "factors": [

        {
            "factor": "decision_making",
            "direction": 1,
            "strength": 1.0,
            "confidence": 0.95
        },

        {
            "factor": "tactical_fit",
            "direction": 1,
            "strength": 0.9,
            "confidence": 0.9
        }

    ]
}


In [128]:
result_a = personalize_player_score(
    "Kevin De Bruyne",
    "Midfielder",
    position_data,
    mock_chatbot_output,
    season=2425
)

result_b = personalize_player_score(
    "Kevin De Bruyne",
    "Midfielder",
    position_data,
    mock_chatbot_output,
    season=2425
)

print_personalized_report(
    result_a
)

print_personalized_report(
    result_b
)


Player: Kevin De Bruyne
Team: Manchester City
Season: 2425
Archetype: Creative / Progressive Midfielder

---------------------------
Baseline Score: 73.37662337662337
Personalized Score: 74.7
Total Adjustment: +1.29

Contextual Factors:

leadership
  Adjustment: +1.71
  Strength: 0.90
  Confidence: 0.95

work_rate
  Adjustment: +1.62
  Strength: 0.90
  Confidence: 0.90

injury_concern
  Adjustment: -2.04
  Strength: 0.80
  Confidence: 0.85

Player: Kevin De Bruyne
Team: Manchester City
Season: 2425
Archetype: Creative / Progressive Midfielder

---------------------------
Baseline Score: 73.37662337662337
Personalized Score: 74.7
Total Adjustment: +1.29

Contextual Factors:

leadership
  Adjustment: +1.71
  Strength: 0.90
  Confidence: 0.95

work_rate
  Adjustment: +1.62
  Strength: 0.90
  Confidence: 0.90

injury_concern
  Adjustment: -2.04
  Strength: 0.80
  Confidence: 0.85


# Actual LLM Implementation

In [129]:
# install LLM
%pip install -U openai pydantic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: C:\Users\Titus\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [130]:

import os
from pydantic import BaseModel, Field, ConfigDict
from typing import Literal
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

print(
    "API key loaded:",
    os.getenv("OPENAI_API_KEY") is not None
)


API key loaded: True


In [131]:
class ScoutingFactor(BaseModel):

    model_config = ConfigDict(
        extra="forbid"
    )

    factor: Literal[
        "leadership",
        "decision_making",
        "work_rate",
        "tactical_fit",
        "press_resistance",
        "adaptability",
        "consistency",
        "injury_concern",
        "communication",
        "off_ball_movement"
    ]

    direction: Literal[-1, 1]

    strength: float = Field(
        ge=0,
        le=1
    )

    confidence: float = Field(
        ge=0,
        le=1
    )

    evidence: str


class ScoutingExtraction(BaseModel):

    model_config = ConfigDict(
        extra="forbid"
    )

    factors: list[ScoutingFactor]

    summary: str

# Give LLM Specific instructions

In [132]:
SCOUTING_SYSTEM_PROMPT = """
You are a soccer scouting information extraction system.

Your job is NOT to rate the player.

Your only job is to convert a user's qualitative scouting
observation into structured contextual factors.

Allowed factors:

leadership
- Ability to lead, organize, influence, or motivate teammates.

decision_making
- Quality of choices made during play.

work_rate
- Effort, intensity, willingness to contribute, and defensive effort.

tactical_fit
- How well the player fits the tactical system described by the user.

press_resistance
- Ability to operate effectively when pressured by opponents.

adaptability
- Ability to adjust to new positions, systems, leagues, or roles.

consistency
- Reliability of performance across games or situations.

injury_concern
- Availability, recurring injuries, or durability concerns.

communication
- Communication and organization with teammates.

off_ball_movement
- Movement, positioning, runs, and spatial awareness without the ball.


RULES:

1. Extract ONLY information supported by the user's text.

2. Do not invent characteristics that the user did not mention.

3. Do not use outside knowledge about the player.

4. Do not calculate a player rating or performance score.

5. Each factor may appear at most once.

6. Omit factors that are not mentioned or reasonably implied.

7. direction must be:
   +1 if the observation improves the player's evaluation.
   -1 if the observation lowers the player's evaluation.

8. strength represents how strongly the user describes the trait:
   0.0 = extremely weak statement
   1.0 = extremely strong statement.

9. confidence represents confidence in YOUR interpretation of the
   user's text, NOT whether the user's claim is factually true.

10. evidence should briefly identify the part of the user's
    observation that supports the factor.

Keep the interpretation conservative.
"""

In [133]:
def extract_factors_with_llm(
    user_text,
    model="gpt-6-luna"
):

    response = client.responses.parse(
        model=model,

        input=[
            {
                "role": "system",
                "content": SCOUTING_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_text
            }
        ],

        text_format=ScoutingExtraction
    )

    if response.output_parsed is None:
        raise ValueError(
            "The model did not return a valid structured response."
        )

    return (
        response.output_parsed
        .model_dump()
    )

In [134]:
scouting_note = """
The player is an excellent leader and makes very good decisions
under pressure. However, I am concerned about his recurring
injury problems.
"""

In [135]:
llm_output = extract_factors_with_llm(
    scouting_note
)

llm_output

{'factors': [{'factor': 'leadership',
   'direction': 1,
   'strength': 0.9,
   'confidence': 0.99,
   'evidence': 'Described as an excellent leader.'},
  {'factor': 'decision_making',
   'direction': 1,
   'strength': 0.8,
   'confidence': 0.98,
   'evidence': 'Makes very good decisions under pressure.'},
  {'factor': 'injury_concern',
   'direction': -1,
   'strength': 0.85,
   'confidence': 0.99,
   'evidence': 'Concern about recurring injury problems.'}],
 'summary': 'The observation highlights strong leadership and decision-making, alongside concern about recurring injuries.'}

In [136]:
validated_factors = validate_factors(
    llm_output
)

print(validated_factors)

[{'factor': 'leadership', 'direction': 1, 'strength': 0.9, 'confidence': 0.99}, {'factor': 'decision_making', 'direction': 1, 'strength': 0.8, 'confidence': 0.98}, {'factor': 'injury_concern', 'direction': -1, 'strength': 0.85, 'confidence': 0.99}]


In [137]:
result = personalize_player_score(
    "Kevin De Bruyne",
    "Midfielder",
    position_data,
    llm_output,
    season=2425
)
print_personalized_report(
    result
)


Player: Kevin De Bruyne
Team: Manchester City
Season: 2425
Archetype: Creative / Progressive Midfielder

---------------------------
Baseline Score: 73.37662337662337
Personalized Score: 74.2
Total Adjustment: +0.83

Contextual Factors:

leadership
  Adjustment: +1.78
  Strength: 0.90
  Confidence: 0.99

decision_making
  Adjustment: +1.57
  Strength: 0.80
  Confidence: 0.98

injury_concern
  Adjustment: -2.52
  Strength: 0.85
  Confidence: 0.99


In [138]:
def evaluate_with_scouting_note(
    player_name,
    position,
    scouting_note,
    position_data,
    season=None,
    model="gpt-6-luna"
):

    parsed_output = (
        extract_factors_with_llm(
            scouting_note,
            model=model
        )
    )

    result = personalize_player_score(
        player_name,
        position,
        position_data,
        parsed_output,
        season=season
    )

    result["scouting_note"] = (
        scouting_note
    )

    result["llm_summary"] = (
        parsed_output["summary"]
    )

    return result

In [139]:


result = evaluate_with_scouting_note(
    "Kevin De Bruyne",
    "Midfielder",
    """
    He reads the game extremely well
    and rarely makes poor decisions.
    """,
    position_data,
    season=2425
)

In [140]:

result = evaluate_with_scouting_note(
    "Cristiano Ronaldo",
    "Forward",
    """
    He remains very decisive in attacking
    situations and shows strong leadership,
    but contributes less defensively.
    """,
    position_data,
    season=2223
)

In [141]:
result = evaluate_with_scouting_note(
    "Virgil van Dijk",
    "Defender",
    """
    He communicates constantly,
    reads danger early, and stays composed
    under pressure.
    """,
    position_data,
    season=2425
)

# Test Cases

In [142]:
evaluation_cases = [
    {
        "note": "He is a vocal leader who constantly organizes teammates.",
        "expected_factors": [
            "leadership",
            "communication"
        ]
    },

    {
        "note": "He struggles when pressed aggressively and often makes poor decisions.",
        "expected_factors": [
            "press_resistance",
            "decision_making"
        ]
    },

    {
        "note": "Recurring injuries have made his availability unreliable.",
        "expected_factors": [
            "injury_concern",
            "consistency"
        ]
    },

    {
        "note": "I watched him play last weekend.",
        "expected_factors": []
    }
]





In [143]:
evaluation_results = []

for i, case in enumerate(evaluation_cases):

    llm_output = extract_factors_with_llm(
        case["note"]
    )

    predicted_factors = {
        item["factor"]
        for item in llm_output["factors"]
    }

    expected_factors = set(
        case["expected_factors"]
    )

    evaluation_results.append({
        "case": i + 1,
        "note": case["note"],
        "expected": expected_factors,
        "predicted": predicted_factors,
        "correct": (
            predicted_factors
            == expected_factors
        )
    })




In [144]:
true_positive = 0
false_positive = 0
false_negative = 0

for result in evaluation_results:


    expected = result["expected"]
    predicted = result["predicted"]

    true_positive += len(
        expected & predicted
    )

    false_positive += len(
        predicted - expected
    )

    false_negative += len(
        expected - predicted
    )



In [145]:
exact_matches = sum(
    result["correct"]
    for result in evaluation_results
)

exact_match_accuracy = (
    exact_matches
    / len(evaluation_results)
)

print(
    "Exact Match Accuracy:",
    round(
        exact_match_accuracy * 100,
        2
    ),
    "%"
)

precision = (
    true_positive
    / (true_positive + false_positive)
    if true_positive + false_positive > 0
    else 0
)

recall = (
    true_positive
    / (true_positive + false_negative)
    if true_positive + false_negative > 0
    else 0
)

f1 = (
    2 * precision * recall
    / (precision + recall)
    if precision + recall > 0
    else 0
)

print(
    f"Precision: {precision:.3f}"
)

print(
    f"Recall: {recall:.3f}"
)

print(
    f"F1 Score: {f1:.3f}"
)

Exact Match Accuracy: 75.0 %
Precision: 1.000
Recall: 0.833
F1 Score: 0.909
